In [1]:
import re
import unicodedata
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import string

In [43]:
import numpy as np

1. Base 1 : Narval

In [2]:
path="C:/Users/CD44/Desktop/Narval/narvalentier.xlsx"

In [3]:
df = pd.read_excel(path) #reading database

In [4]:
df.shape #cheking the shape

(15535, 40)

In [5]:
df.head() #visualise the first 5 rows

,id,nom_objet,numero_dossier,demandeur,nom_institution,site,date_entree,date_sortie,numero_inventaire_interne,datation_objet,...,typologie_objet,date_restitution_photos,date_restitution_fiche_restauration,id_dossier,version,numero_fiche,date_modification,nb_reference,matricule,localisation
0,102037,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,9,000000001,2024-01-09 11:34:14.35,25.0,010393W,NaN
1,102039,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN
2,102038,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN
3,102040,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000004,20 ans,...,NaN,NaN,NaN,4426.0,2,000065,2024-01-09 11:34:14.35,NaN,NaN,NaN
4,100050,NaN,D124-747/98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3619.0,0,1313156,NaN,NaN,NaN,NaN


In [6]:
COLUMNS_TO_MERGE = [  #defining the columns to merge in "tout_info" so the study will be done on only one column
    "materiaux_objet",
    "materiaux_complementaire_objet",
    "description_constat_etat",
    "description_traitement",
    "traitement_demande",
    "conseils_conservation"
]

In [7]:
df["tout_info"] = df[COLUMNS_TO_MERGE].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)

In [8]:
df["Matériaux"]=df[["materiaux_objet","materiaux_complementaire_objet"]].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1)

In [9]:
df.head()

,id,nom_objet,numero_dossier,demandeur,nom_institution,site,date_entree,date_sortie,numero_inventaire_interne,datation_objet,...,date_restitution_fiche_restauration,id_dossier,version,numero_fiche,date_modification,nb_reference,matricule,localisation,tout_info,Matériaux
0,102037,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,4426.0,9,000000001,2024-01-09 11:34:14.35,25.0,010393W,NaN,crêpes,crêpes
1,102039,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes,crêpes
2,102038,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes,crêpes
3,102040,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000004,20 ans,...,NaN,4426.0,2,000065,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes,crêpes
4,100050,NaN,D124-747/98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3619.0,0,1313156,NaN,NaN,NaN,NaN,,


In [10]:
def clean_text(text): 
    if pd.isna(text):
        return text
    
    # Forcer le type string
    text = str(text)
    
    # Minuscules
    text = text.lower()
    
    # Suppression des accents
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    
    # Suppression ponctuation / caractères spéciaux
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Nettoyage espaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [11]:
df["tout_info"] = df["tout_info"].apply(clean_text)
df["Matériaux"] = df["Matériaux"].apply(clean_text)

In [12]:
df["tout_info"]

0        crepes
1        crepes
2        crepes
3        crepes
4              
          ...  
15530          
15531          
15532          
15533          
15534          
Name: tout_info, Length: 15535, dtype: str

In [13]:
df_cuivre = df[df["Matériaux"].str.contains(r"\bcuivre\b|\bcuivreux\b|\bbronze\b|\blaiton\b",
                                          case=False, 
                                          na=False)]
#select only the copper elements

In [14]:
# Keywords organisés en 7 niveaux de poids
# Plus le poids est élevé, plus le mot/terme est fortement lié à la chalconatronite.
keywords = {
    
7:[

"chalco", "chalconatronite"],

6: [
        
"corrosion active",
        "récurrent","cyclique"
        "corrosion pulvérulente"
 "corrosion localisée"
"pulvérulent",
        "pulvérulence","poudre","poudreux"
"chlorure",
"stabilisation chimique",
        "stabilisé"
"sodium",
"sesquicarbonate", "sesquicarbonate de sodium"
    ],

5 :[

	 "égypte", "moyen orient", "époque ptolémaïque","corrosion"],

4 :[
        "fraction active",
        "produit de corrosion bleu-vert",
        "bleu-vert", "bleu vert"
        "bleue verdâtre",
        "sels verts"],

3: [
        "croûte",
        "dépôts blancs"
    ],

2: [
        
        "dégradation",
        "altération",
        "corrosion verte",
        "oxydation",
        
    ],

1: [
        "archéologie", "sculpture", "croyance", "croyances-coutumes",
        "statuette", "statue", "amulette", "monnaie",
        "basse époque","antiquité",
        "pratique votive",
        "fonte à la cire perdue", "coulé", "cire perdue",
        "oxydes d'argent", 
    ]
}
# Familles de synonymes : les termes d'une même famille ne doivent compter qu'une seule fois.
# Exemple : si "sel", "sulfate" et "sesqui" apparaissent dans la même description,
# le score ajouté est 1 seul point, pas 3.
keyword_families = {
    "famille_sel": {
        "weight": 6,
        "terms": [
            "sel",
            "sesquicarbonate", "sesquicarbonate de sodium"
        ]
    },

    "famille_cyclique": {
        "weight": 6,
        "terms": [
            "récurrent", "cyclique"
        ]
    },

    "famille_pulverulence": {
        "weight": 6,
        "terms": [
            "pulvérulent",
            "pulvérulence","poudre","poudreux"
        ]
    },

    "famille_chalco": {
        "weight": 7,
        "terms": [
            "chalco", "chalconatronite"
        ]
    },
    "famille_cire": {
        "weight": 1,
        "terms": [
            "fonte à la cire perdue", "cire perdue"
        ]
    },
    "famille_bleu_vert": {
        "weight": 4,
        "terms": [
            "produit de corrosion bleu-vert", "bleu-vert", "bleu vert"
        ]
    },
    "famille_egypt": {
        "weight": 5,
        "terms": [
            "égypte", "époque ptolémaïque"
        ]
    } 
}


In [15]:
# Normalisation des keywords et des familles avec la même fonction que les descriptions
keywords = {
    k: [clean_text(mot) for mot in liste]
    for k, liste in keywords.items()
}

keyword_families = {
    family_name: {
        "weight": family_data["weight"],
        "terms": [clean_text(term) for term in family_data["terms"]]
    }
    for family_name, family_data in keyword_families.items()
}


In [16]:
keywords

{7: ['chalco', 'chalconatronite'],
 6: ['corrosion active',
  'recurrent',
  'cycliquecorrosion pulverulentecorrosion localiseepulverulent',
  'pulverulence',
  'poudre',
  'poudreuxchlorure',
  'stabilisation chimique',
  'stabilisesodium',
  'sesquicarbonate',
  'sesquicarbonate de sodium'],
 5: ['egypte', 'moyen orient', 'epoque ptolemaique', 'corrosion'],
 4: ['fraction active',
  'produit de corrosion bleu vert',
  'bleu vert',
  'bleu vertbleue verdatre',
  'sels verts'],
 3: ['croute', 'depots blancs'],
 2: ['degradation', 'alteration', 'corrosion verte', 'oxydation'],
 1: ['archeologie',
  'sculpture',
  'croyance',
  'croyances coutumes',
  'statuette',
  'statue',
  'amulette',
  'monnaie',
  'basse epoque',
  'antiquite',
  'pratique votive',
  'fonte a la cire perdue',
  'coule',
  'cire perdue',
  'oxydes d argent']}

In [17]:
# Seuil de détection.
# Avec 7 niveaux de poids, un seuil de 7 signifie :
# - détection directe de "chalco/chalconatronite", ou
# - accumulation de plusieurs indices forts/moyens.
SEUIL = 7


In [18]:
# Générer des variantes proches d'un mot-clé avec une distance d'édition de 1
# Objectif : détecter certaines fautes de frappe simples.
def edits1(word, letters):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    deletes = [L + R[1:] for L, R in splits if R]
    inserts = [L + c + R for L, R in splits for c in letters]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]

    return set(deletes + inserts + replaces)


def expand_with_mapping(keywords, keyword_families=None):
    """
    Crée un mapping : terme détectable -> (canonique, poids, terme_affiche)

    - canonique : sert au calcul du score et évite les doublons.
      Pour une famille, le canonique reste le nom de la famille
      afin que plusieurs synonymes ne soient comptés qu'une seule fois.

    - terme_affiche : sert uniquement à l'affichage dans la colonne mots_detectes.
      Pour une famille, on affiche le terme réellement associé à la famille
      et non le nom technique de la famille.

    Exemple : si "sel" est détecté :
    - canonique = "famille_sel"  -> score ajouté une seule fois
    - terme_affiche = "sel"      -> affichage lisible dans mots_detectes
    """

    all_words = [w for words in keywords.values() for w in words]

    if keyword_families:
        for family_data in keyword_families.values():
            all_words.extend(family_data["terms"])

    letters = set("".join(all_words))
    mapping = {}

    # 1) Mapping des mots-clés classiques
    for weight, words in keywords.items():
        for word in words:
            w = clean_text(word)

            # Pour un mot classique :
            # - canonique = le mot lui-même
            # - affichage = le mot lui-même
            mapping[w] = (w, weight, w)

            # variantes avec distance d'édition 1 seulement pour les mots simples
            # on évite de créer des variantes sur les expressions multi-mots
            if " " not in w:
                for v in edits1(w, letters):
                    if len(v) > 3 and v.isalpha():
                        # La variante détectée pointe vers le mot correct pour l'affichage
                        mapping[v] = (w, weight, w)

    # 2) Mapping des familles synonymes non incrémentables
    if keyword_families:
        for family_name, family_data in keyword_families.items():
            weight = family_data["weight"]

            for term in family_data["terms"]:
                t = clean_text(term)

                # Pour une famille :
                # - canonique = nom de la famille, pour éviter l'incrémentation multiple
                # - affichage = terme détecté de la famille, pas le nom de la famille
                mapping[t] = (family_name, weight, t)

                # variantes avec distance d'édition 1 seulement pour les mots simples
                if " " not in t:
                    for v in edits1(t, letters):
                        if len(v) > 3 and v.isalpha():
                            # La variante détectée pointe vers le terme correct de la famille
                            mapping[v] = (family_name, weight, t)

    return mapping


def get_max_n_from_mapping(mapping):
    """Nombre maximum de mots dans les expressions du mapping."""
    return max(len(term.split()) for term in mapping.keys())


In [19]:
def calculer_score_ngram_canonique(row, colonne, mapping, max_n=None):
    """
    Calcule le score d'une description.

    Points importants :
    - Les expressions longues sont cherchées avant les mots courts.
    - Un même span de texte ne peut pas être réutilisé deux fois.
    - Un même mot canonique ou une même famille canonique ne peut être compté qu'une seule fois.
    - Dans l'affichage, on montre le terme détecté/associé, pas le nom technique de la famille.
    """

    text = row[colonne]

    if pd.isna(text):
        return 0, ""

    tokens = str(text).split()

    if max_n is None:
        max_n = get_max_n_from_mapping(mapping)

    score = 0
    details = []
    used_spans = set()
    seen_canonical = set()

    for n in range(max_n, 0, -1):
        for i in range(len(tokens) - n + 1):

            if any(j in used_spans for j in range(i, i + n)):
                continue

            ngram = " ".join(tokens[i:i + n])

            if ngram in mapping:
                mapping_value = mapping[ngram]

                # Compatibilité avec l'ancien format : (canonique, poids)
                # Nouveau format : (canonique, poids, terme_affiche)
                if len(mapping_value) == 2:
                    canonical_word, weight = mapping_value
                    terme_affiche = canonical_word
                else:
                    canonical_word, weight, terme_affiche = mapping_value

                # Si le mot/famille a déjà été compté, on ne rajoute pas de score.
                if canonical_word in seen_canonical:
                    continue

                score += weight
                details.append(f"'{terme_affiche}', poids {weight}")
                seen_canonical.add(canonical_word)

                for j in range(i, i + n):
                    used_spans.add(j)

    return score, " ; ".join(details)


In [20]:
mapping = expand_with_mapping(keywords, keyword_families)
max_n = get_max_n_from_mapping(mapping)

# .copy() évite les warnings SettingWithCopyWarning de pandas
df_cuivre = df_cuivre.copy()

df_cuivre[["score_chalco", "mots_detectes"]] = df_cuivre.apply(
    lambda row: pd.Series(calculer_score_ngram_canonique(row, "tout_info", mapping, max_n=max_n)),
    axis=1
)


In [21]:
# Détection finale
df_cuivre["est_chalco"] = df_cuivre["score_chalco"] >= SEUIL

# Résultats
print(f"Éléments détectés comme 'chalconatronite' : {df_cuivre['est_chalco'].sum()} / {len(df_cuivre)}")
print(f"Non détectés : {(~df_cuivre['est_chalco']).sum()}")
print(df_cuivre[df_cuivre["est_chalco"]][["score_chalco", "mots_detectes"]].head(10))

Éléments détectés comme 'chalconatronite' : 1193 / 5438
Non détectés : 4245
    score_chalco                                      mots_detectes
7             11     'corrosion', poids 5 ; 'pulverulence', poids 6
9              7       'alteration', poids 2 ; 'corrosion', poids 5
12             7       'alteration', poids 2 ; 'corrosion', poids 5
15            11     'corrosion', poids 5 ; 'pulverulence', poids 6
16            12  'monnaie', poids 1 ; 'sel', poids 6 ; 'corrosi...
22            12  'corrosion', poids 5 ; 'poudre', poids 6 ; 'co...
26             7       'alteration', poids 2 ; 'corrosion', poids 5
29             7             'coule', poids 1 ; 'poudreux', poids 6
35            11         'corrosion', poids 5 ; 'poudreux', poids 6
38             8           'corrosion', poids 5 ; 'croute', poids 3


In [22]:
score_counts = df_cuivre["score_chalco"].value_counts().sort_index()
print(score_counts)

score_chalco
0     1039
1      183
2       28
3       10
4        6
5     2165
6      814
7      112
8       26
9       23
10       3
11     523
12     275
13      74
14      18
15       6
16       3
17      59
18      49
19      14
20       6
23       2
Name: count, dtype: int64


In [23]:
#exporting all columns
df_cuivre.to_excel("cuivre_analyse_narval_7_weigths_test_2.xlsx", index=False)

In [24]:
# Export 3 : seulement les VRAI 
df_true = df_cuivre[df_cuivre["est_chalco"] == True]

df_true.to_excel("resultats_chalco_detectes___analyse_narval_7_weigths_test_2.xlsx", index=False)

2.data2: GPLA

In [25]:
path_="C:/Users/CD44/Desktop/GPLA/GPLA_excel_.xlsx"


In [26]:
df_ = pd.read_excel(path_) #reading database

In [27]:
df_.shape #cheking the shape

(4541, 255)

In [28]:
df_.head() #display the first 5 rows

,Numéro d'inventaire,Numéro de dépôt,Numéro d'inventaire du déposant,Autre numéro,Autre numéro_2,Autre numéro_3,Autre numéro_4,Autre numéro_5,Autre numéro_6,Désignation du bien,...,Etat_4,Intégrité_4.1,Intégrité_4_2,Intégrité_4_3,Date du constat_4,Dégradation_4.1,Multimédia_4,Notice créée le,Notice modifiée le,Institution / Département
0,09/ S.INV,NaN,NaN,N° système v6 : 73425,NaN,NaN,NaN,NaN,NaN,fragment non déterminé,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:15:52,mardi 16 mai 2017 12:54:07,Archéologie militaire/Militaria
1,13/ S.INV,NaN,NaN,N° système v6 : 73429,NaN,NaN,NaN,NaN,NaN,obusier,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:38:33,jeudi 24 juin 2021 08:26:23,Archéologie militaire/Militaria
2,14/ S.INV,NaN,NaN,N° système v6 : 73430,NaN,NaN,NaN,NaN,NaN,obus (paire),...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:02:32,vendredi 17 octobre 2014 10:47:08,Archéologie militaire/Militaria
3,20/ S.INV,NaN,NaN,N° système v6 : 73437,NaN,NaN,NaN,NaN,NaN,épée de ville,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:30:05,lundi 27 octobre 2014 10:51:49,Archéologie militaire/Militaria
4,22/ S.INV,NaN,NaN,N° système v6 : 73439,NaN,NaN,NaN,NaN,NaN,fourreau d'épée,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:34:21,lundi 27 octobre 2014 12:52:37,Archéologie militaire/Militaria


In [29]:
COLUMNS_TO_MERGE_ = [  #defining the columns to merge in "tout_info" so the study will be done on only one column
    "Désignation du bien",
    "Désignation du bien_4",
    "Désignation du bien_2",
    "Matière",
    "Matière_2",
    "Matière_3","Matière_4","Matière_5","Matière_6","Matière_7","Matière_8","Matière_9",
    "Technique","Technique_2","Technique_3","Technique_4","Technique_5","Technique_6","Technique_7","Technique_9","Technique_8","Technique_10","Technique_11",
    "Matière_2_2","Matière_2_3","Matière_2_4",'Matière_2_5',
 'Matière_2_6',
 'Matière_3.1',
 'Matière_3_2',
 'Matière_3_3',
 'Matière_4.1',
 'Matière_4_2',
 'Matière_5.1',
 'Matière_5_2',
 'Matière_6.1',
 'Matière_6_2',
 'Matière_7.1','Technique','Technique_2','Technique_4','Technique_5','Technique_6','Technique_7','Technique_8','Technique_9','Technique_10','Technique_11','Technique_2.1','Technique_2_2','Technique_2_3','Technique_2_4','Technique_2_5','Technique_2_6','Technique_2_7','Technique_3_2','Technique_3_3','Technique_3_4','Technique_4.1','Technique_4_2','Technique_5.1','Technique_5_2','Technique_6.1',
 'Technique_6_2','Technique_3','Technique_3.1','Dégradation',
 'Dégradation_2',
 'Dégradation_3',
 'Dégradation_4',
 'Dégradation_5',
 'Dégradation_6',
 'Dégradation_7',
 'Dégradation_8',
 'Dégradation_2_3',
 'Dégradation_2_4',
 'Dégradation_2_5',
 'Dégradation_2_6',
 'Dégradation_3.1',
 'Dégradation_3_2',
 'Dégradation_3_3',"Intervention requise",'Intervention requise_2',
 'Intervention requise_3',
 'Intervention requise_4',
 'Intervention requise_5',
 'Intervention requise_6',
 'Intervention requise_7',
 'Intervention requise_8',
 'Intervention requise_9',
 'Intervention requise_10',
 'Intervention requise_2.1',
 'Intervention requise_2_2',
 'Intervention requise_2_3',
 'Intervention requise_2_4',
 'Intervention requise_2_5',
 'Intervention requise_2_6',
 'Intervention requise_2_7',
 'Intervention requise_2_8',
 'Intervention requise_2_9',
 'Intervention requise_3.1',
 'Intervention requise_3_2',
 'Intervention requise_3_3',
 'Intervention requise_3_4',
 'Intervention requise_3_5',"Description analytique", "Description analytique_2"
]

clm_mat=["Matière",
    "Matière_2",
    "Matière_3","Matière_4","Matière_5","Matière_6","Matière_7","Matière_8","Matière_9",
    "Technique","Technique_2","Technique_3","Technique_4","Technique_5","Technique_6","Technique_7","Technique_9","Technique_8","Technique_10","Technique_11",
    "Matière_2_2","Matière_2_3","Matière_2_4",'Matière_2_5',
 'Matière_2_6',
 'Matière_3.1',
 'Matière_3_2',
 'Matière_3_3',
 'Matière_4.1',
 'Matière_4_2',
 'Matière_5.1',
 'Matière_5_2',
 'Matière_6.1',
 'Matière_6_2',
 'Matière_7.1']

In [30]:
df_["tout_info_"] = df_[COLUMNS_TO_MERGE_].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)
df_["Matériaux_"] = df_[clm_mat].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)

In [31]:
df_.head()

,Numéro d'inventaire,Numéro de dépôt,Numéro d'inventaire du déposant,Autre numéro,Autre numéro_2,Autre numéro_3,Autre numéro_4,Autre numéro_5,Autre numéro_6,Désignation du bien,...,Intégrité_4_2,Intégrité_4_3,Date du constat_4,Dégradation_4.1,Multimédia_4,Notice créée le,Notice modifiée le,Institution / Département,tout_info_,Matériaux_
0,09/ S.INV,NaN,NaN,N° système v6 : 73425,NaN,NaN,NaN,NaN,NaN,fragment non déterminé,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:15:52,mardi 16 mai 2017 12:54:07,Archéologie militaire/Militaria,fragment non déterminé . cuivre jaune . liège ...,cuivre jaune . liège
1,13/ S.INV,NaN,NaN,N° système v6 : 73429,NaN,NaN,NaN,NaN,NaN,obusier,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:38:33,jeudi 24 juin 2021 08:26:23,Archéologie militaire/Militaria,obusier . bronze . bois . corrosion : traces ....,bronze . bois
2,14/ S.INV,NaN,NaN,N° système v6 : 73430,NaN,NaN,NaN,NaN,NaN,obus (paire),...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:02:32,vendredi 17 octobre 2014 10:47:08,Archéologie militaire/Militaria,obus (paire) . cuivre . forme cylindrique Mêm...,cuivre
3,20/ S.INV,NaN,NaN,N° système v6 : 73437,NaN,NaN,NaN,NaN,NaN,épée de ville,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:30:05,lundi 27 octobre 2014 10:51:49,Archéologie militaire/Militaria,épée de ville . alliage ferreux . or . . Parti...,alliage ferreux . or
4,22/ S.INV,NaN,NaN,N° système v6 : 73439,NaN,NaN,NaN,NaN,NaN,fourreau d'épée,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:34:21,lundi 27 octobre 2014 12:52:37,Archéologie militaire/Militaria,fourreau d'épée . cuir . . Garnitures de laito...,cuir


In [32]:
df_["tout_info_"] = df_["tout_info_"].apply(clean_text)

In [33]:
df_["tout_info_"]

0       fragment non determine cuivre jaune liege piec...
1       obusier bronze bois corrosion traces depot art...
2       obus paire cuivre forme cylindrique meme dimen...
3       epee de ville alliage ferreux or partiellement...
4       fourreau d epee cuir garnitures de laiton bout...
                              ...                        
4536    figurine minerve en course alliage cuivreux mo...
4537    epee alliage ferreux laiton forge forge oxydat...
4538    poignard alliage cuivreux usure oxydation cass...
4539    manche de couteau alliage cuivreux moule moule...
4540    instrument de chirurgie alliage cuivreux moule...
Name: tout_info_, Length: 4541, dtype: str

In [34]:
df_cuivre_ = df_[df_["Matériaux_"].str.contains(r"\bcuivre\b|\bcuivreux\b|\bbronze\b|\blaiton\b",
                                          case=False, 
                                          na=False)]
#select only the copper elements

In [35]:
df_cuivre_

,Numéro d'inventaire,Numéro de dépôt,Numéro d'inventaire du déposant,Autre numéro,Autre numéro_2,Autre numéro_3,Autre numéro_4,Autre numéro_5,Autre numéro_6,Désignation du bien,...,Intégrité_4_2,Intégrité_4_3,Date du constat_4,Dégradation_4.1,Multimédia_4,Notice créée le,Notice modifiée le,Institution / Département,tout_info_,Matériaux_
0,09/ S.INV,NaN,NaN,N° système v6 : 73425,NaN,NaN,NaN,NaN,NaN,fragment non déterminé,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:15:52,mardi 16 mai 2017 12:54:07,Archéologie militaire/Militaria,fragment non determine cuivre jaune liege piec...,cuivre jaune . liège
1,13/ S.INV,NaN,NaN,N° système v6 : 73429,NaN,NaN,NaN,NaN,NaN,obusier,...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:38:33,jeudi 24 juin 2021 08:26:23,Archéologie militaire/Militaria,obusier bronze bois corrosion traces depot art...,bronze . bois
2,14/ S.INV,NaN,NaN,N° système v6 : 73430,NaN,NaN,NaN,NaN,NaN,obus (paire),...,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:02:32,vendredi 17 octobre 2014 10:47:08,Archéologie militaire/Militaria,obus paire cuivre forme cylindrique meme dimen...,cuivre
6,56.614,NaN,NaN,N° système v6 : 8673,NaN,NaN,NaN,NaN,NaN,manche,...,NaN,NaN,NaT,NaN,NaN,vendredi 13 décembre 1996 00:00:00,vendredi 9 janvier 2015 17:24:59,Archéologie militaire/Militaria,manche bronze encrassement trace s de chocs a ...,bronze
7,56.841,NaN,NaN,N° système v6 : 8882,NaN,NaN,NaN,NaN,NaN,clef,...,NaN,NaN,NaT,NaN,NaN,vendredi 13 décembre 1996 00:00:00,vendredi 9 juillet 2010 16:35:51,Archéologie,clef alliage cuivreux martele battu martele ba...,alliage cuivreux . martelé-battu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4536,NaN,D 2004.2.1 (restitué),NaN,N° système v6 : 79798,NaN,NaN,NaN,NaN,NaN,figurine,...,NaN,NaN,NaT,NaN,NaN,mercredi 16 juin 2004 09:39:41,jeudi 26 septembre 2024 16:07:42,Archéologie,figurine minerve en course alliage cuivreux mo...,alliage cuivreux . moulé
4537,NaN,D 2010.2.12,NaN,N° système v6 : 92233,NaN,NaN,NaN,NaN,NaN,épée,...,NaN,NaN,NaT,NaN,NaN,lundi 15 juin 2015 17:22:52,jeudi 22 avril 2021 15:01:29,Archéologie,epee alliage ferreux laiton forge forge oxydat...,alliage ferreux . laiton . forgé
4538,NaN,D 2019.3.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,poignard,...,NaN,NaN,NaT,NaN,NaN,lundi 9 mars 2020 16:47:03,mercredi 11 janvier 2023 12:47:11,Archéologie,poignard alliage cuivreux usure oxydation cass...,alliage cuivreux
4539,NaN,D 2022.2.2,MT 1267.0003,NaN,NaN,NaN,NaN,NaN,NaN,manche de couteau,...,NaN,NaN,NaT,NaN,NaN,lundi 5 décembre 2022 12:26:05,mercredi 15 janvier 2025 18:19:39,Archéologie,manche de couteau alliage cuivreux moule moule...,alliage cuivreux . moulé


In [36]:
mapping = expand_with_mapping(keywords, keyword_families)
max_n = get_max_n_from_mapping(mapping)

# .copy() évite les warnings SettingWithCopyWarning de pandas
df_cuivre_ = df_cuivre_.copy()

df_cuivre_[["score_chalco", "mots_detectes"]] = df_cuivre_.apply(
    lambda row: pd.Series(calculer_score_ngram_canonique(row, "tout_info_", mapping, max_n=max_n)),
    axis=1
)


In [37]:
# Détection finale
df_cuivre_["est_chalco"] = df_cuivre_["score_chalco"] >= SEUIL

# Résultats
print(f"Éléments détectés comme 'chalconatronite' : {df_cuivre_['est_chalco'].sum()} / {len(df_cuivre_)}")
print(f"Non détectés : {(~df_cuivre_['est_chalco']).sum()}")
print(df_cuivre_[df_cuivre_["est_chalco"]][["score_chalco", "mots_detectes"]].head(10))

Éléments détectés comme 'chalconatronite' : 492 / 4487
Non détectés : 3995
    score_chalco                                      mots_detectes
16            12  'stabilisation chimique', poids 6 ; 'coule', p...
23             7  'stabilisation chimique', poids 6 ; 'coule', p...
31            12  'stabilisation chimique', poids 6 ; 'coule', p...
33             7  'stabilisation chimique', poids 6 ; 'coule', p...
34            11  'stabilisation chimique', poids 6 ; 'corrosion...
35            11  'stabilisation chimique', poids 6 ; 'corrosion...
44            12  'stabilisation chimique', poids 6 ; 'coule', p...
50            12  'corrosion active', poids 6 ; 'stabilisation c...
51            12  'stabilisation chimique', poids 6 ; 'corrosion...
52            11  'stabilisation chimique', poids 6 ; 'corrosion...


In [38]:
score_counts_ = df_cuivre_["score_chalco"].value_counts().sort_index()
print(score_counts_)

score_chalco
0     1427
1     1862
2       97
3       29
4        9
5      350
6      221
7       57
8       27
9       17
10      31
11      80
12     169
13      14
14      10
15       3
16      29
17       9
18       7
19       3
20      34
22       1
23       1
Name: count, dtype: int64


In [39]:
#exporting all columns
df_cuivre_.to_excel("cuivre_gpla_7_weigths_test2.xlsx", index=False)

In [40]:
# Export 3 : seulement les VRAI 
df_true_ = df_cuivre_[df_cuivre_["est_chalco"] == True]

df_true_.to_excel("resultats_chalco_detectes___analyse_gpla_7_weigths_test2.xlsx", index=False)

 GPLA=> 4487 / 4541  cuivre || 380 / 4487 chalco  |||||||  
 NARVAL => 5438 / 15535  cuivre || 702 / 5438 chalco

In [41]:
o

NameError: name 'o' is not defined

In [ ]:
import re
import unicodedata
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import string
import numpy as np   ### AJOUT

# =========================
# NARVAL
# =========================

path="C:/Users/CD44/Desktop/Narval/narvalentier.xlsx"
df = pd.read_excel(path)

COLUMNS_TO_MERGE = [
"materiaux_objet",
"materiaux_complementaire_objet",
"description_constat_etat",
"description_traitement",
"traitement_demande",
"conseils_conservation"
]

df["tout_info"] = df[COLUMNS_TO_MERGE].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)

df["Matériaux"]=df[["materiaux_objet","materiaux_complementaire_objet"]].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1)

def clean_text(text):
    if pd.isna(text):
        return text  

df["tout_info"] = df["tout_info"].apply(clean_text)
df["Matériaux"] = df["Matériaux"].apply(clean_text)

df_cuivre = df[df["Matériaux"].str.contains(
    r"\bcuivre\b|\bcuivreux\b|\bbronze\b|\blaiton\b",
    case=False,
    na=False
)]

# =========================
# SCORE CHALCO EXISTANT
# =========================

mapping = expand_with_mapping(keywords, keyword_families)
max_n = get_max_n_from_mapping(mapping)

df_cuivre = df_cuivre.copy()

df_cuivre[["score_chalco", "mots_detectes"]] = df_cuivre.apply(
    lambda row: pd.Series(calculer_score_ngram_canonique(row, "tout_info", mapping, max_n=max_n)),
    axis=1
)

df_cuivre["est_chalco"] = df_cuivre["score_chalco"] >= SEUIL


In [44]:


# ============================================================
# ✅ AJOUT NARVAL : MULTI TRAITEMENT PAR DOUBLON INVENTAIRE
# ============================================================

if "numero_inventaire_interne" in df_cuivre.columns:
    counts = df_cuivre["numero_inventaire_interne"].value_counts()
    df_cuivre["multi_traitement_narval"] = df_cuivre["numero_inventaire_interne"].map(counts) > 1
else:
    df_cuivre["multi_traitement_narval"] = False


# ============================================================
# ✅ AJOUT SCORE +6
# ============================================================

POIDS_MULTI = 6

df_cuivre["score_chalco"] = df_cuivre["score_chalco"] + np.where(
    df_cuivre["multi_traitement_narval"],
    POIDS_MULTI,
    0
)


# =========================
# EXPORT NARVAL
# =========================

df_cuivre.to_excel("cuivre_analyse_narval_7_weigths_test_22.xlsx", index=False)

df_true = df_cuivre[df_cuivre["est_chalco"] == True]
df_true.to_excel("resultats_chalco_detectes___analyse_narval_7_weigths_test_22.xlsx", index=False)




In [48]:

# ============================================================
# ✅ AJOUT GPLA : restauration + multi date
# ============================================================

# restauration
df_cuivre_["motif_norm"] = df_cuivre_["Motif du constat d'état"].str.lower().fillna("")

df_cuivre_["flag_restauration"] = df_cuivre_["motif_norm"].str.contains(
    r"restaurat",
    na=False,
    regex=True
)

# colonnes date auto
date_cols = [col for col in df_cuivre_.columns if "Date du constat" in col.lower()]

df_cuivre_["nb_dates"] = df_cuivre_[date_cols].notna().sum(axis=1)

df_cuivre_["multi_traitement_gpla"] = (
    (df_cuivre_["nb_dates"] > 1) &
    (df_cuivre_["flag_restauration"])
)


# ============================================================
# ✅ AJOUT SCORE +6 GPLA
# ============================================================

df_cuivre_["score_chalco"] = df_cuivre_["score_chalco"] + np.where(
    df_cuivre_["multi_traitement_gpla"],
    POIDS_MULTI,
    0
)


# =========================
# EXPORT GPLA
# =========================

df_cuivre_.to_excel("cuivre_gpla_7_weigths_test22.xlsx", index=False)

df_true_ = df_cuivre_[df_cuivre_["est_chalco"] == True]
df_true_.to_excel("resultats_chalco_detectes___analyse_gpla_7_weigths_test22.xlsx", index=False)